In [ ]:
# CELL 1: Install Libraries
!pip install natasha transformers spacy
# Download the Russian spaCy model
!python -m spacy download ru_core_news_sm

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 54.8 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=8cd42bde1fce98181c32036e56e76481c31bafac1f5a10006221517e594fe100
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
  Created wheel for intervaltree: filename=intervaltree-3.1.0-py2.py3-none-any.whl size=26098 sha256=118df5df9ab3259376eacc350886087d3b3b1c3b03a2cb946e0e4162d303a92f
  Stored in directory: /root/.cache/pip/wheels/65/c3/c3/238bf93c243597857edd94ddb0577faa74a8e16e9585896e83
Successfully built docopt intervaltree
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Load All 20,000 Sentences
import nltk
from datasets import load_dataset

print("Loading WikiANN Dataset...")
dataset = load_dataset("wikiann", "ru")

# 1. Define the Tag Map
tags_map = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}

# 2. Helper function
def convert_to_nltk_format(hf_data):
    formatted_data = []
    for item in hf_data:
        tokens = item['tokens']
        ner_tags = item['ner_tags']
        sentence = []
        for token, tag_id in zip(tokens, ner_tags):
            tag_label = tags_map[tag_id]
            sentence.append((token, tag_label))
        formatted_data.append(sentence)
    return formatted_data

# 3. Use the FULL splits
# Previous code: dataset['train'].select(range(5000))
# New code: dataset['train'] (All 20,000 sentences)
print("Formatting Training Data (20k sentences)...")
train_set_ner = convert_to_nltk_format(dataset['train'])

print("Formatting Test Data...")
# We'll use the full validation set (10,000 sentences) for testing
# If this is too slow to evaluate later, we can slice it then.
test_set_ner = convert_to_nltk_format(dataset['validation'])

print(f"Data Ready: {len(train_set_ner)} training sentences, {len(test_set_ner)} test sentences.")

Loading WikiANN Dataset...


README.md: 0.00B [00:00, ?B/s]

ru/validation-00000-of-00001.parquet:   0%|          | 0.00/809k [00:00<?, ?B/s]

ru/test-00000-of-00001.parquet:   0%|          | 0.00/816k [00:00<?, ?B/s]

ru/train-00000-of-00001.parquet:   0%|          | 0.00/1.63M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Formatting Training Data (20k sentences)...
Formatting Test Data...
Data Ready: 20000 training sentences, 10000 test sentences.


In [ ]:
# Train HMM for NER & Calculate F1 Score
!pip install sklearn-crfsuite

from nltk.tag import HiddenMarkovModelTagger
from sklearn_crfsuite import metrics
import warnings
warnings.filterwarnings("ignore")

print("Training HMM on NER data...")
# Train on the full 20k sentences
hmm_ner = HiddenMarkovModelTagger.train(train_set_ner)

print("Calculating F1 Score on test set...")

# 2. Prepare Inputs and Targets
# We need to separate the words from the tags to run F1 scoring
test_tokens = [[t[0] for t in sent] for sent in test_set_ner]
y_true = [[t[1] for t in sent] for sent in test_set_ner]

# 3. Get Predictions
# tag_sents() allows us to tag the whole test list at once
pred_output = hmm_ner.tag_sents(test_tokens)
y_pred = [[t[1] for t in sent] for sent in pred_output]

# 4. Define Labels for Grading
# We grab all unique tags found in the data, but remove 'O' to avoid inflating the score
labels = list(set([tag for sent in y_true for tag in sent]))
if 'O' in labels:
    labels.remove('O')
sorted_labels = sorted(labels)

# 5. Calculate F1
f1 = metrics.flat_f1_score(y_true, y_pred, average='weighted', labels=sorted_labels)
print(f"HMM F1 Score: {f1 * 100:.2f}%")

# 6. Tricky Sentence Test
tricky_sent = "Илон Маск купил Твиттер"
tokens = tricky_sent.split()
tags = hmm_ner.tag(tokens)

print("-" * 40)
print(f"Testing Tricky Sentence: '{tricky_sent}'")
print(f"{'WORD':<12} {'PREDICTED TAG'}")
print("-" * 40)

for word, tag in tags:
    # O = Outside (Not an entity)
    print(f"{word:<12} {tag}")

Training HMM on NER data...
Calculating F1 Score on test set...
HMM F1 Score: 73.91%
----------------------------------------
Testing Tricky Sentence: 'Илон Маск купил Твиттер'
WORD         PREDICTED TAG
----------------------------------------
Илон         B-PER
Маск         I-PER
купил        I-PER
Твиттер      I-PER


In [ ]:
# Train CRF for NER & Calculate F1 Score

import sklearn_crfsuite
from sklearn_crfsuite import metrics

# 2. Define Features (The "Brain" of the model)
def word2features_ner(sentence, i):
    word = sentence[i][0]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),  # Capitalized? (Important for Names)
        'word.isdigit()': word.isdigit(),
    }

    # Context: Previous Word
    if i > 0:
        word1 = sentence[i-1][0]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
        })
    else:
        features['BOS'] = True # Beginning of Sentence

    # Context: Next Word
    if i < len(sentence)-1:
        word1 = sentence[i+1][0]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
        })
    else:
        features['EOS'] = True # End of Sentence

    return features

def sent2features(sent):
    return [word2features_ner(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

# 3. Format Data
print("Extracting Features for CRF...")
# Prepare Training Data
X_train_ner = [sent2features(s) for s in train_set_ner]
y_train_ner = [sent2labels(s) for s in train_set_ner]

# Prepare Test Data (Required for F1 Calculation)
X_test_ner = [sent2features(s) for s in test_set_ner]
y_test_ner = [sent2labels(s) for s in test_set_ner]

# 4. Train
print("Training CRF...")
crf_ner = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf_ner.fit(X_train_ner, y_train_ner)
print("CRF Training Complete!")

# 5. Calculate F1 Score
print("Calculating F1 Score...")
y_pred = crf_ner.predict(X_test_ner)

# Remove 'O' (Outside) tag so we don't inflate the score
labels = list(crf_ner.classes_)
if 'O' in labels:
    labels.remove('O')

f1 = metrics.flat_f1_score(y_test_ner, y_pred, average='weighted', labels=labels)
print(f"CRF F1 Score: {f1 * 100:.2f}%")

# 6. Test the Tricky Sentence
print("-" * 40)
tricky_sent = "Илон Маск купил Твиттер"
print(f"Testing Tricky Sentence: '{tricky_sent}'")

# We have to fake a 'sentence' structure for the features
tokens = tricky_sent.split()
dummy_input = [(word, 'N/A') for word in tokens]
crf_features = [sent2features(dummy_input)]

prediction = crf_ner.predict(crf_features)[0]

print(f"{'WORD':<12} {'PREDICTED TAG'}")
print("-" * 40)
for word, tag in zip(tokens, prediction):
    print(f"{word:<12} {tag}")

Extracting Features for CRF...
Training CRF...
CRF Training Complete!
Calculating F1 Score...
CRF F1 Score: 85.13%
----------------------------------------
Testing Tricky Sentence: 'Илон Маск купил Твиттер'
WORD         PREDICTED TAG
----------------------------------------
Илон         B-ORG
Маск         I-ORG
купил        I-ORG
Твиттер      I-ORG


In [ ]:
# Fine-Tune BERT for NER
!pip install accelerate seqeval evaluate

import os
# Force-disable W&B logging so it doesn't ask for login
os.environ["WANDB_DISABLED"] = "true"

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

# 2. Load Data (WikiANN Russian)
print("Loading WikiANN Data...")
dataset = load_dataset("wikiann", "ru")

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

# Map tags
label_list = dataset["train"].features[f"ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# 3. Load Tokenizer
model_checkpoint = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 4. Alignment Function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Aligning data...")
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_eval = eval_dataset.map(tokenize_and_align_labels, batched=True)

# 5. Load Model
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# 6. Metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 7. Train (With logging disabled)
args = TrainingArguments(
    "bert-ner-russian",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    report_to="none" # Second safeguard to disable logging
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting Training...")
trainer.train()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6d546019465310a9b2228e946cc73ab0155042f7296ebd855bec95a85ea16252
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
Loading WikiANN Data...
Aligning data...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Starting Training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.172100,0.137302,0.885349,0.903438,0.894302,0.961582
2,0.101300,0.138536,0.903687,0.912113,0.907880,0.964641
3,0.067700,0.143289,0.907008,0.918113,0.912527,0.967088


TrainOutput(global_step=3750, training_loss=0.1326378423055013, metrics={'train_runtime': 1173.9265, 'train_samples_per_second': 51.111, 'train_steps_per_second': 3.194, 'total_flos': 791054530114464.0, 'train_loss': 0.1326378423055013, 'epoch': 3.0})

In [ ]:
# The Final NER Test
import torch

def test_my_bert(sentence):
    print(f"Testing: '{sentence}'")

    # Ensure model is in eval mode
    model.eval()

    # Tokenize raw text
    # We send it to the same device as the model (GPU)
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)

    # Get predictions
    with torch.no_grad():
        logits = model(**inputs).logits

    predictions = torch.argmax(logits, dim=2)

    # Decode
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = predictions[0].cpu().numpy()

    print(f"{'TOKEN':<15} {'PREDICTED TAG'}")
    print("-" * 35)

    for token, label_id in zip(tokens, labels):
        # Skip special tokens
        if token in ['[CLS]', '[SEP]']: continue

        # Get readable label
        label = id2label[label_id]

        # Clean up subwords (##) for display
        clean_token = token.replace("##", "")

        print(f"{clean_token:<15} {label}")
# The sentence that broke the CRF
test_my_bert("Илон Маск купил Твиттер")

Testing: 'Илон Маск купил Твиттер'
TOKEN           PREDICTED TAG
-----------------------------------
Ил              B-PER
он              I-PER
Маск            I-PER
купил           O
Твит            B-ORG
тер             I-ORG


In [ ]:
# CELL 31 (FIXED & ROBUST): Final Visual Comparison & Strict F1 Scoring
import pandas as pd
import torch
from sklearn_crfsuite import metrics
from transformers import AutoTokenizer

# --- 0. SAFETY CHECK: Reload Tokenizer & Model ---
print("Ensuring model and tokenizer are loaded...")
checkpoint = "DeepPavlov/rubert-base-cased"
# We reload the tokenizer (safe to do, it's just a dictionary)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# We ensure 'model' points to the trained version
# If 'trainer' exists (from Cell 28), we use its model.
# If 'model' exists but 'trainer' doesn't, we keep 'model'.
if 'trainer' in globals():
    model = trainer.model
elif 'model' not in globals():
    raise ValueError("ERROR: The BERT model is missing. Please re-run CELL 28 (Training) first!")

# --- 1. VISUAL COMPARISON ON 3 SENTENCES ---

# 1. Define Sentences
test_sentences = [
    "Газпром находится в Москве",       # Standard
    "Илон Маск купил Твиттер",          # Tricky: The "Verb Barrier"
    "Иван читает роман в Париже"        # Tricky: Ivan (Name) vs roman (book)
]

# 2. Prediction Helpers
def get_hmm_pred(tokens):
    return [tag for word, tag in hmm_ner.tag(tokens)]

def get_crf_pred(tokens):
    dummy_input = [(word, 'N/A') for word in tokens]
    features = [sent2features(dummy_input)]
    return crf_ner.predict(features)[0]

def get_bert_pred(original_tokens):
    inputs = tokenizer(original_tokens, is_split_into_words=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()

    word_ids = inputs.word_ids()
    aligned_tags = []
    previous_word_idx = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None: continue
        if word_idx != previous_word_idx:
            tag_id = predictions[i]
            label = id2label[tag_id]
            aligned_tags.append(label)
            previous_word_idx = word_idx
    return aligned_tags

# 3. Print The Visual Table (WITH BIO TAGS)
print(f"{'WORD':<12} {'HMM':<10} {'CRF':<10} {'BERT (Winner)':<15}")
print("=" * 60)

for sent in test_sentences:
    print(f"\nInput: {sent}")
    print("-" * 60)
    tokens = sent.split()

    h_tags = get_hmm_pred(tokens)
    c_tags = get_crf_pred(tokens)
    b_tags = get_bert_pred(tokens)

    for i, word in enumerate(tokens):
        b_tag = b_tags[i] if i < len(b_tags) else "???"

        # Highlight Differences
        prefix = "->" if (b_tag != c_tags[i] or b_tag != h_tags[i]) else "  "

        print(f"{prefix}{word:<11} {h_tags[i]:<10} {c_tags[i]:<10} {b_tag:<15}")

# --- PART 2: LIVE F1 CALCULATION ---
print("\n" + "="*60)
print("CALCULATING FINAL F1 SCORES ON TEST SET... (Please Wait)")
print("="*60)

# Define labels (Ignore 'O')
target_labels = list(crf_ner.classes_)
if 'O' in target_labels: target_labels.remove('O')

# 1. HMM F1 Score
print("Scoring HMM...")
hmm_test_tokens = [[t[0] for t in sent] for sent in test_set_ner]
hmm_true_tags = [[t[1] for t in sent] for sent in test_set_ner]
hmm_preds = hmm_ner.tag_sents(hmm_test_tokens)
hmm_pred_tags = [[t[1] for t in sent] for sent in hmm_preds]
hmm_f1 = metrics.flat_f1_score(hmm_true_tags, hmm_pred_tags, average='weighted', labels=target_labels) * 100

# 2. CRF F1 Score
print("Scoring CRF...")
X_test_ner_final = [sent2features(s) for s in test_set_ner]
y_test_ner_final = [sent2labels(s) for s in test_set_ner]
y_pred_crf = crf_ner.predict(X_test_ner_final)
crf_f1 = metrics.flat_f1_score(y_test_ner_final, y_pred_crf, average='weighted', labels=target_labels) * 100

# 3. RuBERT F1 Score
print("Scoring BERT...")
# Use the Trainer to get the exact number
bert_metrics = trainer.evaluate()
bert_f1 = bert_metrics['eval_f1'] * 100

# --- PART 3: THE FINAL SCOREBOARD ---
print("\n" + "="*35)
print("FINAL LEADERBOARD (Strict F1)")
print("="*35)
print(f"1. RuBERT (Fine-Tuned):  {bert_f1:.2f}% (Winner)")
print(f"2. CRF (Feature-Based):  {crf_f1:.2f}%")
print(f"3. HMM (Statistical):    {hmm_f1:.2f}%")

Ensuring model and tokenizer are loaded...
WORD         HMM        CRF        BERT (Winner)  

Input: Газпром находится в Москве
------------------------------------------------------------
->Газпром     B-ORG      O          B-ORG          
  находится   O          O          O              
  в           O          O          O              
  Москве      B-LOC      B-LOC      B-LOC          

Input: Илон Маск купил Твиттер
------------------------------------------------------------
->Илон        B-PER      B-ORG      B-PER          
->Маск        I-PER      I-ORG      I-PER          
->купил       I-PER      I-ORG      O              
->Твиттер     I-PER      I-ORG      B-ORG          

Input: Иван читает роман в Париже
------------------------------------------------------------
->Иван        B-ORG      O          B-PER          
->читает      I-ORG      O          O              
->роман       I-ORG      O          O              
  в           O          O          O            


FINAL LEADERBOARD (Strict F1)
1. RuBERT (Fine-Tuned):  91.25% (Winner)
2. CRF (Feature-Based):  85.13%
3. HMM (Statistical):    73.91%
